install 

In [ ]:
!pip install pysam pyBigWig scipy numpy requests

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!wget -O "/content/drive/MyDrive/Chromogen_Project/data/raw/hg38.fa.gz" "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz"
!gunzip "/content/drive/MyDrive/Chromogen_Project/data/raw/hg38.fa.gz"

In [ ]:
import os

# القاموس بعد إصلاح الفواصل، علامات التنصيص، وتكملة الرابط المقطوع لـ H1-hESC
cells_to_download = {
    "GM12878": "https://www.encodeproject.org/files/ENCFF960FMM/@@download/ENCFF960FMM.bigWig",
    "K562": "https://www.encodeproject.org/files/ENCFF078EJM/@@download/ENCFF078EJM.bigWig",
    "HepG2": "https://www.encodeproject.org/files/ENCFF854HNI/@@download/ENCFF854HNI.bigWig",
    "H1-hESC": "https://www.encodeproject.org/files/ENCFF462HTM/@@download/ENCFF462HTM.bigWig"
}

target_dir = "/content/drive/MyDrive/Chromogen_Project/data/raw/"
os.makedirs(target_dir, exist_ok=True)

for cell_name, url in cells_to_download.items():
    file_path = os.path.join(target_dir, f"{cell_name}_DNase.bigWig")

    if not os.path.exists(file_path):
        print(f"📥 جاري تحميل ملف الـ DNase لخلية {cell_name} إلى الدرايف...")
        !wget -q -O "{file_path}" "{url}"
        print(f"✅ تم تحميل {cell_name} بنجاح!")
    else:
        print(f"⏩ الملف {cell_name}_DNase.bigWig موجود مسبقاً، لا داعي للتحميل.")

In [ ]:
import subprocess
import os
import sys
import shutil
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import gaussian_filter

dnase_drive_paths = {
    "GM12878": "/content/drive/MyDrive/Chromogen_Project/data/raw/GM12878_DNase.bigWig",
    "K562": "/content/drive/MyDrive/Chromogen_Project/data/raw/K562_DNase.bigWig",
    "HepG2": "/content/drive/MyDrive/Chromogen_Project/data/raw/HepG2_DNase.bigWig",
    "H1-hESC": "/content/drive/MyDrive/Chromogen_Project/data/raw/H1-hESC_DNase.bigWig"
}
fasta_path = '/content/drive/MyDrive/Chromogen_Project/data/raw/hg38.fa'

hic_online_urls = {
    "GM12878": "https://www.encodeproject.org/files/ENCFF017AAT/@@download/ENCFF017AAT.hic",
    "K562": "https://www.encodeproject.org/files/ENCFF621AIY/@@download/ENCFF621AIY.hic",
    "HepG2": "https://www.encodeproject.org/files/ENCFF244AVN/@@download/ENCFF244AVN.hic",
    "H1-hESC": "https://www.encodeproject.org/files/ENCFF104THR/@@download/ENCFF104THR.hic"
}

def initialize_environment():
    print("[INFO] البدء في فحص وتثبيت الاعتمادات الجينومية...")
    dependencies = ['pyBigWig', 'pyfaidx', 'hictkpy', 'numpy', 'pandas', 'torch', 'scipy']
    for dep in dependencies:
        try:
            __import__(dep)
        except ImportError:
            print(f" 🔄 جاري تثبيت المكتبة: {dep}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", dep, "--quiet"])
    print("[INFO] تم التحقق من تثبيت كافة المكتبات بنجاح.")

initialize_environment()

import pyBigWig
from pyfaidx import Fasta
import hictkpy

class StrawCLIWrapper:
    def __init__(self, binary_dir="./straw_bin"):
        self.binary_dir = binary_dir
        self.binary_path = os.path.join(binary_dir, "straw")
        self._compile_from_source()

    def _compile_from_source(self):
        if os.path.exists(self.binary_path):
            print(f"[INFO] أداة straw مجمعة بالفعل ومتاحة في: {self.binary_path}")
            return

        print("[COMPILE] لم يتم العثور على أداة straw مجمعة. البدء في التنزيل والتجميع من المصدر...")
        os.makedirs(self.binary_dir, exist_ok=True)
        temp_src_dir = "./straw_src"
        if os.path.exists(temp_src_dir):
            shutil.rmtree(temp_src_dir)

        try:
            subprocess.run(["git", "clone", "https://github.com/aidenlab/straw.git", temp_src_dir], check=True, stdout=subprocess.DEVNULL)
            cpp_src_path = os.path.join(temp_src_dir, "C++")
            print("[COMPILE] جاري تشغيل المترجم g++ لبناء الأداة الثنائية...")
            subprocess.run([
                "g++", "-std=c++11", "-O3", "-o", self.binary_path,
                os.path.join(cpp_src_path, "main.cpp"),
                os.path.join(cpp_src_path, "straw.cpp"),
                "-lcurl", "-lz"
            ], check=True)
            print(f" ✅ تم تجميع أداة straw بنجاح وحفظها في: {self.binary_path}")
        except Exception as e:
            print(f" ❌ فشلت عملية تجميع أداة straw من المصدر: {str(e)}")
        finally:
            if os.path.exists(temp_src_dir):
                shutil.rmtree(temp_src_dir)

    def extract_sparse_matrix(self, hic_file, chrom, start, end, resolution, norm="NONE"):
        chrom = str(chrom)
        alt_chrom = chrom.replace("chr", "") if chrom.startswith("chr") else f"chr{chrom}"
        region = f"{chrom}:{start}:{end}"

        cmd = [self.binary_path, "observed", norm, hic_file, region, region, "BP", str(resolution)]

        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
            if result.stdout.strip():
                return result.stdout

            alt_region = f"{alt_chrom}:{start}:{end}"
            cmd[4], cmd[5] = alt_region, alt_region
            result_alt = subprocess.run(cmd, capture_output=True, text=True, check=True)
            return result_alt.stdout
        except subprocess.CalledProcessError as e:
            try:
                alt_region = f"{alt_chrom}:{start}:{end}"
                cmd[4], cmd[5] = alt_region, alt_region
                result_alt = subprocess.run(cmd, capture_output=True, text=True, check=True)
                return result_alt.stdout
            except Exception:
                print(f"[ERROR] فشل استخراج البيانات بكلا الصيغتين. رسالة الخطأ: {e.stderr}")
                return None

class HiCContactReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.cli_wrapper = StrawCLIWrapper()

    def fetch_dense_matrix(self, chrom, start, end, resolution, norm="NONE"):
        import io
        num_bins = (end - start) // resolution
        raw_output = self.cli_wrapper.extract_sparse_matrix(self.file_path, chrom, start, end, resolution, norm)

        # 🔥 التعديل الذكي: مصفوفة مابينج على الديسك المحلي لكولاب لمنع استهلاك الـ RAM
        # تم استخدام النمط 'w+' الصحيح تماماً لتجنب أي خطأ
        filename = f"/content/temp_hic_{chrom}.dat"
        if os.path.exists(filename):
            try: os.remove(filename)
            except: pass
            
        # إنشاء الملف على الديسك مباشرة
        matrix = np.memmap(filename, dtype=np.float32, mode='w+', shape=(num_bins, num_bins))
        matrix[:] = 0.0  # تهيئة المصفوفة بالأصفار مباشرة على الهارد ديسك بدون لمس الرام

        if not raw_output or not raw_output.strip():
            return matrix

        # قراءة الداتا دفعة واحدة بـ Pandas ومحرك C الذكي
        df = pd.read_csv(
            io.StringIO(raw_output),
            sep='\t',
            names=['pos_x', 'pos_y', 'count'],
            dtype={'pos_x': np.int32, 'pos_y': np.int32, 'count': np.float32},
            na_values=['nan']
        ).fillna(0.0)

        # تحويل الإحداثيات لفهارس (Bins) دفعة واحدة عمليات شعاعية (Vectorized)
        df['idx_x'] = (df['pos_x'] - start) // resolution
        df['idx_y'] = (df['pos_y'] - start) // resolution

        # التأكد من بقاء الفهارس داخل الحدود الصالحة
        df = df[(df['idx_x'] < num_bins) & (df['idx_y'] < num_bins) & (df['idx_x'] >= 0) & (df['idx_y'] >= 0)]

        # بناء وتعبئة المصفوفة المتناظرة مباشرة على القرص الصلب بسرعة البرق
        x_vals = df['idx_x'].values
        y_vals = df['idx_y'].values
        counts = df['count'].values

        matrix[x_vals, y_vals] = counts
        matrix[y_vals, x_vals] = counts
        
        matrix.flush()  # دفع وتأكيد حفظ كافة البيانات على الهارد ديسك
        return matrix
class GenomicIntegrationDataset(Dataset):
    def __init__(self, fasta_path, bigwig_path, hic_path, chrom,
                 resolution=20000, window_size=1280000, stride=200000,
                 density_threshold=0.1, gaussian_sigma=0.5):
        self.fasta_path = fasta_path
        self.bigwig_path = bigwig_path
        self.hic_path = hic_path
        self.chrom = chrom
        self.resolution = resolution
        self.window_size = window_size
        self.stride = stride
        self.density_threshold = density_threshold
        self.gaussian_sigma = gaussian_sigma
        
        self.genome = Fasta(fasta_path)
        self.fasta_chrom = chrom if chrom in self.genome else (chrom.replace("chr", "") if chrom.startswith("chr") else f"chr{chrom}")
        self.chrom_len = len(self.genome[self.fasta_chrom])
        
        self.hic_reader = HiCContactReader(hic_path)

        # ⚡⚡ [تحسين السرعة الخارق للـ Hi-C]: جلب مصفوفة الكروموسوم كاملة بالـ C-engine
        print(f"[⚡ OPTIMIZATION] جاري جلب مصفوفة الـ Hi-C للكروموسوم {chrom} بالكامل وتخزينها...")
        hic_chrom_input = chrom.replace("chr", "") if chrom.startswith("chr") else chrom
        self.full_hic_matrix = self.hic_reader.fetch_dense_matrix(hic_chrom_input, 0, self.chrom_len, self.resolution, norm="NONE")
        print("✅ تم تحميل المصفوفة الكاملة في الذاكرة بنجاح!")

        # ⚡⚡ [تحسين السرعة الخارق للـ DNase]: كاش كامل في الذاكرة لمنع الـ Deadlocks أثناء التدريب
        print(f"[⚡ OPTIMIZATION] جاري سحب إشارة DNase للكروموسوم {chrom} بالكامل...")
        num_hic_bins = self.full_hic_matrix.shape[0]
        
        with pyBigWig.open(self.bigwig_path) as bw:
            bw_chrom = chrom if chrom in bw.chroms() else (chrom.replace("chr", "") if chrom.startswith("chr") else f"chr{chrom}")
            try:
                dnase_vals = bw.stats(bw_chrom, 0, self.chrom_len, type="mean", nBins=num_hic_bins)
                self.full_dnase_array = np.array([v if (v is not None and not np.isnan(v)) else 0.0 for v in dnase_vals], dtype=np.float32)
            except Exception as e:
                print(f"⚠️ فشل جلب DNase بالكامل، سيتم تعبئته بأصفار: {e}")
                self.full_dnase_array = np.zeros(num_hic_bins, dtype=np.float32)
        print("✅ تم تحميل إشارة DNase الكاملة وإغلاق اتصال الملف بأمان.")

        # حساب وتصفية النوافذ الصالحة
        self.windows = []
        for start in range(0, self.chrom_len - self.window_size, self.stride):
            end = start + self.window_size
            idx_start = start // self.resolution
            idx_end = end // self.resolution

            local_matrix = self.full_hic_matrix[idx_start:idx_end, idx_start:idx_end]
            density = np.count_nonzero(local_matrix) / local_matrix.size if local_matrix.size > 0 else 0.0

            if density >= self.density_threshold:
                self.windows.append((start, end))

        print(f" 📦 تم العثور على {len(self.windows)} نافذة جينومية صالحة للتدريب.")

    def __len__(self):
        return len(self.windows)

    def _one_hot_encode_dna(self, sequence):
        mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3, 'a': 0, 'c': 1, 'g': 2, 't': 3}
        encoding = np.zeros((4, len(sequence)), dtype=np.float32)
        for i, char in enumerate(sequence):
            idx = mapping.get(char, -1)
            if idx != -1: encoding[idx, i] = 1.0
            else: encoding[:, i] = 0.25
        return encoding

    def __getitem__(self, idx):
        start, end = self.windows[idx]
        if idx % 100 == 0:
            print(f"🔄 [PROGRESS] جاري معالجة النافذة رقم {idx}/{len(self.windows)} | الإحداثيات: {self.chrom}:{start}-{end}")
        
        # 1. الـ DNA
        dna_seq = self.genome[self.fasta_chrom][start:end].seq
        dna_tensor = torch.from_numpy(self._one_hot_encode_dna(dna_seq))

        idx_start = start // self.resolution
        idx_end = end // self.resolution
        num_bins = self.window_size // self.resolution

        # 2. قطع الـ DNase مباشرة من الميموري (سريع وآمن 100% مع الـ multi-workers)
        dnase_arr = self.full_dnase_array[idx_start:idx_end].copy()
        if dnase_arr.shape[0] != num_bins:
            padded_dnase = np.zeros(num_bins, dtype=np.float32)
            padded_dnase[:min(dnase_arr.shape[0], num_bins)] = dnase_arr[:min(dnase_arr.shape[0], num_bins)]
            dnase_arr = padded_dnase
        dnase_tensor = torch.from_numpy(dnase_arr)

        # 3. قطع الـ Hi-C من الميموري
        hic_matrix = self.full_hic_matrix[idx_start:idx_end, idx_start:idx_end].copy()
        if hic_matrix.shape[0] != num_bins or hic_matrix.shape[1] != num_bins:
            padded_matrix = np.zeros((num_bins, num_bins), dtype=np.float32)
            min_r, min_c = min(hic_matrix.shape[0], num_bins), min(hic_matrix.shape[1], num_bins)
            padded_matrix[:min_r, :min_c] = hic_matrix[:min_r, :min_c]
            hic_matrix = padded_matrix

        # 4. التصفية والتطبيع اللوغاريتمي الموضعي
        density = np.count_nonzero(hic_matrix) / hic_matrix.size if hic_matrix.size > 0 else 0.0
        if density < self.density_threshold:
            hic_matrix = np.zeros_like(hic_matrix)
        else:
            if self.gaussian_sigma > 0:
                hic_matrix = gaussian_filter(hic_matrix, sigma=self.gaussian_sigma)
            hic_matrix = np.log1p(hic_matrix)

        hic_tensor = torch.from_numpy(hic_matrix).to(torch.float32)

        return {"dna": dna_tensor, "dnase": dnase_tensor, "hic": hic_tensor}



In [ ]:
def run_pipeline_on_google_drive(fasta_drive_path, bigwig_drive_path, hic_url_or_path, chrom="chr22", resolution=5000):
    try:
        from google.colab import drive
        print("🔄 جاري محاولة الاتصال بـ Google Drive لفحص ملفات الفاستا والـ DNase...")
        drive.mount('/content/drive')
        print("✅ تم الاتصال بالقرص السحابي الخاص بك بنجاح.")
    except ImportError:
        print("⚠️ تشغيل الكود خارج Colab. تخطي تركيب قرص Google Drive.")

    if not os.path.exists(fasta_drive_path) or not os.path.exists(bigwig_drive_path):
        print("❌ خطأ: تأكدي من وجود ملفات الفاستا والـ BigWig المحتفظة بالدرايف في المسارات الصحيحة.")
        return None

    print(f"🌐 سيتم جلب بيانات Hi-C مباشرة من الرابط أونلاين بنظام الكاش الذكي...")

    # حساب الـ window_size والـ stride بناءً على الـ resolution الممرر ديناميكياً
    window_size = 256 * resolution
    stride = window_size // 4

    dataset = GenomicIntegrationDataset(
        fasta_path=fasta_drive_path,
        bigwig_path=bigwig_drive_path,
        hic_path=hic_url_or_path,
        chrom=chrom,
        resolution=resolution,
        window_size=window_size,
        stride=stride,
        density_threshold=0.1,
        gaussian_sigma=0.5
    )

    dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
    print(f"🎉 خط البيانات جاهز ومحسّن تماماً الآن!")
    return dataloader

# ///////////////////////////////////////////////////////////////////////////////////

def run_for_cell(cell_type, chrom="chr22", resolution=5000):
    """
    تختار ملفات الدرايف المحلية للـ DNase وتجلب رابط الـ Hi-C الأونلاين للخلية المطلوبة مع تمرير الـ resolution
    """
    dnase_path = dnase_drive_paths.get(cell_type)
    hic_url = hic_online_urls.get(cell_type)

    if not dnase_path:
        print(f"❌ خطأ: مسار ملف الـ DNase غير موجود للخلية {cell_type} على الدرايف.")
        return None

    if not hic_url:
        print(f"❌ خطأ: لا يوجد رابط Hi-C معرف للخلية {cell_type}.")
        return None

    return run_pipeline_on_google_drive(fasta_path, dnase_path, hic_url, chrom=chrom, resolution=resolution)

In [ ]:

import h5py
import io
def save_dataset_to_hdf5_chrom(dataset, cell_type, chrom, split):
    import os
    import numpy as np
    import h5py
    from scipy.ndimage import gaussian_filter

    base_dir = "/content/drive/MyDrive/Chromogen_Project/data/processed"
    save_path = os.path.join(base_dir, split, cell_type)
    os.makedirs(save_path, exist_ok=True)
    full_file_path = os.path.join(save_path, f"{chrom}_hic.h5")

    print(f"📂 البدء في بناء ملف HDF5 الذكي للكروموسوم {chrom}...")

    try:
        num_bins = dataset.full_hic_matrix.shape[0]
        sigma = getattr(dataset, 'gaussian_sigma', 0)
        has_gaussian = sigma > 0
        
        # حساب الهامش (Padding) المطلوب فوق وتحت البلوك لضمان دقة الفلتر الجاوسي 2D
        # رياضياً 4 أضعاف سيجما كافية ليكون الخطأ معدوم تماماً
        pad = int(4 * sigma) + 1 if has_gaussian else 0

        print("💾 جاري ضغط البيانات وتطبيق الفلتر الجاوسي واللوغاريتم تدفقاً (سطر بسطر)...")
        dnase_processed = dataset.full_dnase_array.copy().astype(np.float16)

        with h5py.File(full_file_path, 'w') as f:
            # حجز مساحة المصفوفة في HDF5
            hic_dataset = f.create_dataset(
                'hic', 
                shape=(num_bins, num_bins), 
                dtype='f2', 
                compression='gzip', 
                compression_opts=4, 
                chunks=True
            )
            
            chunk_size = 2000
            for i in range(0, num_bins, chunk_size):
                end_i = min(i + chunk_size, num_bins)
                
                if has_gaussian:
                    # 1. جلب البلوك مع الهوامش من الديسك للرام
                    start_pad = max(0, i - pad)
                    end_pad = min(num_bins, end_i + pad)
                    
                    # سحب البلوك للرام (حجمه آمن جداً ~400 ميجا ولن يسبب كراش)
                    v_chunk = dataset.full_hic_matrix[start_pad:end_pad, :].copy()
                    
                    # 2. تطبيق الفلتر ثنائي الأبعاد بالرام بسرعة البرق (أجزاء من الثانية)
                    v_chunk_filtered = gaussian_filter(v_chunk, sigma=sigma)
                    
                    # 3. التخلص من الهوامش وأخذ الأسطر الحقيقية المستهدفة فقط
                    local_start = i - start_pad
                    local_end = local_start + (end_i - i)
                    row_chunk = v_chunk_filtered[local_start:local_end, :].copy()
                else:
                    # إذا لم يكن هناك فلتر، نسحب الأسطر مباشرة
                    row_chunk = dataset.full_hic_matrix[i:end_i, :].copy()
                
                # تطبيق اللوغاريتم
                row_chunk = np.log1p(row_chunk)
                
                # تصفير النصف السفلي يدوياً للبلوك الحالي (المثلث العلوي فقط) لضغط الحجم
                for r_idx in range(i, end_i):
                    row_chunk[r_idx - i, :r_idx] = 0.0
                
                # سكب النتيجة فوراً بالملف النهائي وتحويلها لـ float16
                hic_dataset[i:end_i, :] = row_chunk.astype(np.float16)

            # حفظ باقي المصفوفات والبيانات الوصفية
            f.create_dataset('dnase', data=dnase_processed, dtype='f2', compression='gzip', compression_opts=4, chunks=True)
            f.create_dataset('windows', data=np.array(dataset.windows, dtype=np.int32), dtype='i4')
            
            f.attrs['resolution']  = dataset.resolution
            f.attrs['window_size'] = dataset.window_size
            f.attrs['chrom']       = chrom  
            f.attrs['cell_type']   = cell_type

        size_mb = os.path.getsize(full_file_path) / 1e6
        print(f"🎉 تم الحفظ بنجاح ميكروسكوبي! المسار: {full_file_path} | الحجم الشامل: {size_mb:.1f} MB")
        return full_file_path

    except Exception as e:
        print(f"❌ فشل حفظ ملف الكروموسوم {chrom}: {str(e)}")
        return None
        
    finally:
        # تنظيف نهائي للـ memmap الأصلي من الرام والديسك لتوفير المساحة
        if hasattr(dataset, 'full_hic_matrix') and isinstance(dataset.full_hic_matrix, np.memmap):
            base_filename = dataset.full_hic_matrix.filename
            del dataset.full_hic_matrix
            if os.path.exists(base_filename):
                try: os.remove(base_filename)
                except: pass


للتشغيل 

In [ ]:
if __name__ == "__main__":

    # ══════════════════════════════════════════════
    #  اختاري الخلية (شيلي التعليق عن الواحدة)
    # ══════════════════════════════════════════════
    # cell_to_test = "H1-hESC"
    # cell_to_test = "GM12878"
    cell_to_test = "K562"
    # cell_to_test = "HepG2"

    # ══════════════════════════════════════════════
    #  اختاري الكروموسوم والـ Split
    # ══════════════════════════════════════════════
    # ------------------ [ Train ] ------------------
    # chrom_to_test = "chr1";  split = "train"
    # chrom_to_test = "chr2";  split = "train"
    # chrom_to_test = "chr5";  split = "train"
    # chrom_to_test = "chr8";  split = "train"
    # chrom_to_test = "chr11"; split = "train"
    chrom_to_test = "chr13"; split = "train"
    # chrom_to_test = "chr14"; split = "train"
    # chrom_to_test = "chr15"; split = "train"
    # chrom_to_test = "chr16"; split = "train"
    # chrom_to_test = "chr17"; split = "train"
    # ------------------ [ Val ] --------------------
    # chrom_to_test = "chr18"; split = "val"
    # chrom_to_test = "chr19"; split = "val"
    # ------------------ [ Test ] -------------------
    # chrom_to_test = "chr10"; split = "test"
    # chrom_to_test = "chr21"; split = "test"
    # chrom_to_test = "chr22"; split = "test"

    # ══════════════════════════════════════════════
    #  2. إعدادات الـ Resolution الثابتة (5Kb)
    # ══════════════════════════════════════════════
    resolution  = 5_000         # دقة ثابتة 5Kb
    window_size = 1_280_000     # حجم النافذة (256 بن)
    stride      = 320_000       # مقدار القفزة (التداخل 75%)

    # ══════════════════════════════════════════════
    #  3. تحقق إذا الملف موجود مسبقاً لمنع التكرار
    # ══════════════════════════════════════════════
    save_dir = f"/content/drive/MyDrive/Chromogen_Project/data/processed/{split}/{cell_to_test}"
    out_path = f"{save_dir}/{chrom_to_test}_hic.h5"

    if os.path.exists(out_path):
        print(f"⏭️  الملف موجود مسبقاً — تخطي:\n➡️  {out_path}")
    else:
        print(f"🚀 {cell_to_test} | {chrom_to_test} [{split.upper()}] | {resolution//1000}kb resolution")

        # الاستدعاء النظيف تماماً والمطابق للكلاس المحدث
        dataset = GenomicIntegrationDataset(
            fasta_path=fasta_path,
            bigwig_path=dnase_drive_paths[cell_to_test],
            hic_path=hic_online_urls[cell_to_test],
            chrom=chrom_to_test,
            resolution=resolution,
            window_size=window_size,
            stride=stride,
            density_threshold=0.1,
            gaussian_sigma=0.5
        )

        if len(dataset.windows) == 0:
            print(f"⚠️  لا نوافذ صالحة في {chrom_to_test} — تخطي.")
        else:
            os.makedirs(save_dir, exist_ok=True)
            save_dataset_to_hdf5_chrom(dataset, cell_to_test, chrom_to_test, split)

        # تنظيف الذاكرة العشوائية فوراً
        del dataset
        import gc; gc.collect()
        print("🧹 تم تنظيف الذاكرة العشوائية بنجاح.")

فحص اذا الملف والويندوز الموجودة 

In [ ]:

import os
import h5py
import numpy as np

def scan_all_hdf5_windows_for_data(file_path):
    """
    تابع ذكي متوافق مع صيغة HDF5 يقوم بمسح شامل لكافة النوافذ
    ويكشف بدقة النوافذ التي تحتوي على بيانات حقيقية للـ DNase والـ Hi-C.
    """
    if not os.path.exists(file_path):
        print(f"❌ خطأ: لم يتم العثور على الملف في المسار المحدد:\n➡️ {file_path}")
        return

    print("🔍 جاري فحص وبث كافة النوافذ بداخل ملف HDF5 بحثاً عن الإشارات الحية...")
    
    try:
        # فتح ملف HDF5 للقراءة
        with h5py.File(file_path, 'r') as f:
            # 1. سحب الإحداثيات والإعدادات
            windows = f['windows'][:]
            resolution = f.attrs['resolution']
            
            total_windows = len(windows)
            active_windows = []
            
            # 2. تحميل المصفوفات الكاملة للذاكرة لمرة واحدة لتسريع الفحص الصاروخي
            dnase_all = f['dnase'][:]
            hic_all = f['hic'][:]
            
            # المرور على كل النوافذ المخزنة
            for idx in range(total_windows):
                start, end = windows[idx]
                
                # تحويل الإحداثيات الجينومية إلى فهارس مصفوفة (Bins)
                idx_start = start // resolution
                idx_end = end // resolution
                
                # اقتطاع الجزء الخاص بالنافذة الحالية
                dnase_slice = dnase_all[idx_start:idx_end]
                hic_slice = hic_all[idx_start:idx_end, idx_start:idx_end]
                
                # الفحص المنطقي باستخدام numpy
                dnase_has_data = np.any(dnase_slice != 0)
                hic_has_data = np.any(hic_slice != 0)
                
                # حساب القيم العظمى
                dnase_max = np.max(dnase_slice) if len(dnase_slice) > 0 else 0.0
                hic_max = np.max(hic_slice) if hic_slice.size > 0 else 0.0
                
                # إذا كانت النافذة تحتوي على بيانات، نقوم بتسجيلها
                if dnase_has_data or hic_has_data:
                    active_windows.append({
                        "idx": idx,
                        "range": f"{start}-{end}",
                        "dnase": "✅ حية" if dnase_has_data else "❌ أصفار",
                        "dnase_max": float(dnase_max),
                        "hic": "✅ حية" if hic_has_data else "❌ أصفار",
                        "hic_max": float(hic_max)
                    })
                    
    except Exception as e:
        print(f"❌ فشل في قراءة أو تحليل ملف HDF5: {str(e)}")
        return

    print(f"\n✨ اكتمل الفحص الشامل لـ {total_windows} نافذة جينية بسلام!")
    print("-" * 85)
    
    if not active_windows:
        print("⚠️ صدمة! الملف بأكمله يحتوي على أصفار فقط في كل النوافذ.")
        print("💡 هذا يعني أن الكروموسوم بأكمله تم إنشاؤه من منطقة فجوة غير مقروءة.")
    else:
        print(f"🎉 وجدنا {len(active_windows)} نافذة تحتوي على إشارات حية ونشطة بيولوجياً!\n")
        # طباعة الجدول المنسق الرهيب تبعك
        print(f"{'Index':<7} | {'Genomic Region (chr22)':<25} | {'DNase Signal':<14} | {'Hi-C Matrix':<14}")
        print("-" * 85)
        for w in active_windows:
            print(f"{w['idx']:<7} | {w['range']:<25} | {w['dnase']:<10} ({w['dnase_max']:.4f}) | {w['hic']:<10} ({w['hic_max']:.4f})")
            
        print("\n💡 الخطوة التالية:")
        print(f"الملف شغال 100% ومضغوط الحجم! خدي أي Index فيه الـ Hi-C حية وجربيه بـ Dataset التدريب.")

# -------------------------------------------------------------
# التشغيل
path_to_file = "/content/drive/MyDrive/Chromogen_Project/data/processed/H1-hESC/chr22.h5"
scan_all_hdf5_windows_for_data(file_path=path_to_file)

فحص اذا رابط ال hic يعمل 

In [ ]:
import requests

# الرابط المراد فصحه
url = "https://www.encodeproject.org/files/ENCFF104THR/@@download/ENCFF104THR.hic"

print("📡 جاري فحص الرابط والتأكد من استجابة السيرفر ومحتوى الملف...")
try:
    # نطلب أول 500 بايت فقط من الملف للتأكد من سلامته دون تحميله بالكامل
    headers = {"Range": "bytes=0-500"}
    response = requests.get(url, headers=headers, timeout=15)
    
    print(f"\n📊 رمز حالة السيرفر (HTTP Status): {response.status_code}")
    print(f"📄 نوع المحتوى المرجَع (Content-Type): {response.headers.get('Content-Type')}\n")
    
    if response.status_code in [200, 206]:
        content = response.content
        print(f"🧬 أول 20 بايت من الملف (Raw Bytes): {content[:20]}")
        
        # التحقق من وجود توقيع ملف الـ Hi-C السحري
        if content.startswith(b"HIC"):
            print("✅ الرابط شغال تماماً! والملف هو ملف Hi-C حقيقي ويحتوي على الـ Magic String السليم.")
        else:
            print("❌ تحذير: الملف لا يبدأ بـ 'HIC'. هذا ليس ملف بيانات جينومية حقيقي حالياً!")
            if b"<Error>" in content or b"<?xml" in content:
                print("\n⚠️ السيرفر أرجع رسالة خطأ نصية (XML) من Amazon S3. إليكِ نص الخطأ لمعرفته:")
                print("-" * 50)
                print(response.text)
                print("-" * 50)
            else:
                print("\nالمحتوى المرجَع غريب وغير معروف:")
                print(response.text[:300])
    else:
        print(f"❌ السيرفر رفض الطلب تماماً وأرجع رمز الخطأ: {response.status_code}")
        print("محتوى الرد:")
        print(response.text[:500])

except Exception as e:
    print(f"❌ فشل الاتصال بالرابط نهائياً. السبب: {str(e)}")

بيرسم ال hi-c , dnase وبيعطي معلومات عن الملف 

In [ ]:
import os
import h5py
import matplotlib.pyplot as plt
import numpy as np
from pyfaidx import Fasta

def _one_hot_encode_dna_local(sequence):
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3, 'a': 0, 'c': 1, 'g': 2, 't': 3}
    seq_len = len(sequence)
    encoding = np.zeros((4, seq_len), dtype=np.float32)
    for i, char in enumerate(sequence):
        idx = mapping.get(char, -1)
        if idx != -1:
            encoding[idx, i] = 1.0
        else:
            encoding[:, i] = 0.25
    return encoding
def load_verify_and_plot_from_hdf5_optimized(file_path, fasta_path, window_idx=0):
    if not os.path.exists(file_path):
        print(f"❌ خطأ: لم يتم العثور على ملف HDF5 في المسار المحدد:\n➡️ {file_path}")
        return
    if not os.path.exists(fasta_path):
        print(f"❌ خطأ: لم يتم العثور على ملف الفاستا في المسار المحدد:\n➡️ {fasta_path}")
        return

    print(f"📂 فتح ملف الـ HDF5 الذكي واقتطاع النافذة [{window_idx}]...")

    try:
        with h5py.File(file_path, 'r') as f:
            if 'windows' not in f or 'dnase' not in f:
                print("❌ خطأ: الملف غير متوافق مع البنية الجديدة.")
                return

            # طباعة معلومات الأبعاد الخام المخزنة في الملف للتشخيص
            print(f"🔍 [DEBUG] أبعاد مصفوفة النوافذ في الملف: {f['windows'].shape}")
            print(f"🔍 [DEBUG] نوع بيانات النوافذ: {f['windows'].dtype}")
            
            # قراءة سطر النافذة المطلوبة فقط بشكل آمن
            window_raw = f['windows'][window_idx]
            print(f"🔍 [DEBUG] القيم الخام المستخرجة للنافذة [{window_idx}]: {window_raw}")

            # تحويل القيم صراحة إلى int64 لمنع الـ Overflow أثناء القراءة
            start = int(window_raw[0])
            end = int(window_raw[1])
            
            resolution = int(f.attrs['resolution'])
            window_size = int(f.attrs['window_size'])
            chrom_name = f.attrs.get('chrom', 'chr5')
            cell_type = f.attrs.get('cell_type', 'Unknown')
            num_bins = window_size // resolution

            print(f"📌 الإحداثيات بعد التحويل الصريح: Start={start}, End={end}, Resolution={resolution}")

            # حساب الفهارس بشكل آمن تماماً باستخدام عزل القيمة الصحيحة
            idx_start = int(np.floor(start / resolution))
            idx_end = int(np.floor(end / resolution))
            
            print(f"📊 فهارس الـ Bins المحسوبة للقطع: idx_start={idx_start}, idx_end={idx_end}")

            # 1. جلب الـ DNA محلياً من ملف الـ Fasta
            genome = Fasta(fasta_path)
            fasta_chrom = chrom_name if chrom_name in genome else (chrom_name.replace("chr", "") if chrom_name.startswith("chr") else f"chr{chrom_name}")
            
            # حماية في حال كانت الإحداثيات المستخرجة خارج حدود الكروموسوم الفعلي في فاستا
            chrom_len_fasta = len(genome[fasta_chrom])
            safe_end = min(end, chrom_len_fasta)
            dna_seq = genome[fasta_chrom][start:safe_end].seq
            dna_np = _one_hot_encode_dna_local(dna_seq)

            # 2. قراءة إشارة الـ DNase من الملف
            dnase_ds = f['dnase']
            print(f"🔍 [DEBUG] حجم مصفوفة DNase الكاملة في الملف: {dnase_ds.shape}")
            
            # حماية حدود المصفوفة
            safe_idx_end = min(idx_end, dnase_ds.shape[0])
            dnase_slice = dnase_ds[idx_start:safe_idx_end].astype(np.float32)
            
            if dnase_slice.shape[0] != num_bins:
                padded_dnase = np.zeros(num_bins, dtype=np.float32)
                min_len = min(dnase_slice.shape[0], num_bins)
                padded_dnase[:min_len] = dnase_slice[:min_len]
                dnase_slice = padded_dnase

            # 3. اقتطاع مصفوفة الـ Hi-C
            # 3. اقتطاع مصفوفة الـ Hi-C
            hic_ds = f['hic']
            print(f"🔍 [DEBUG] حجم مصفوفة Hi-C الكاملة في الملف: {hic_ds.shape}")
            
            safe_hic_end = min(idx_end, hic_ds.shape[0])
            
            # ⚡ [الإصلاح الجوهري]: تحويل القطعة فوراً إلى مصفوفة نيمباي مستقلة تماماً في الذاكرة
            hic_slice = np.array(hic_ds[idx_start:safe_hic_end, idx_start:safe_hic_end], dtype=np.float32)

            if hic_slice.shape[0] != num_bins or hic_slice.shape[1] != num_bins:
                padded_hic = np.zeros((num_bins, num_bins), dtype=np.float32)
                min_r = min(hic_slice.shape[0], num_bins)
                min_c = min(hic_slice.shape[1], num_bins)
                padded_hic[:min_r, :min_c] = hic_slice[:min_r, :min_c]
                hic_slice = padded_hic

            # ⚡ [طريقة متناظرة آمنة وموفرة للذاكرة لمنع Overflow الـ int32]:
            hic_full = np.zeros_like(hic_slice)
            # دمج المثلث العلوي والسفلي بدون عمليات ضرب فهارس معقدة
            hic_full = np.maximum(hic_slice, hic_slice.T) 

            print("✅ تم تحميل جميع المكونات وعزلها في الذاكرة بنجاح! جاري الرسم الآمن...")

            # 4. الرسم البياني الآمن
            fig, axes = plt.subplots(1, 2, figsize=(16, 6))
            
            # استخدام interpolation="nearest" أو "none" صريحة وتحديد النطاق لمنع كراش matplotlib
            im = axes[0].imshow(hic_full, cmap="Reds", origin="lower", interpolation="nearest")
            axes[0].set_title(f"Hi-C Contact Matrix ({num_bins}x{num_bins} Bins)", fontweight='bold')
            fig.colorbar(im, ax=axes[0])

            axes[1].plot(dnase_slice, color="royalblue", linewidth=2)
            axes[1].fill_between(range(len(dnase_slice)), dnase_slice, color="royalblue", alpha=0.3)
            axes[1].set_title(f"DNase-seq Signal ({num_bins} Bins)", fontweight='bold')
            axes[1].grid(True, linestyle="--", alpha=0.5)

            plt.suptitle(f"Chromogen Inspector | Region: {chrom_name}:{start}-{end}", fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()

    except Exception as e:
        print(f"❌ فشل في قراءة أو رسم مكونات النافذة: {str(e)}")

# -------------------------------------------------------------
# طريقة التشغيل المحدثة والمنظمة:
fasta_path = '/content/drive/MyDrive/Chromogen_Project/data/raw/hg38.fa'
path_to_file = "/content/drive/MyDrive/Chromogen_Project/data/processed/train/H1-hESC/chr6_hic.h5" 

# استدعاء دالة الفحص بدون الحاجة لتمرير ملف BigWig خارجي!
load_verify_and_plot_from_hdf5_optimized(
    file_path=path_to_file, 
    fasta_path=fasta_path, 
    window_idx=100
)